In [2]:
import os
import pandas as pd
import subprocess as sp
import polars as pl

In [3]:
out_dir = "../../results/04_single_cell_access_atac/20_split_fragment_by_marker_scores"
os.makedirs(out_dir, exist_ok=True)

In [9]:
df_fragments = pl.read_csv(
            "../../results/04_single_cell_access_atac/09_merge_fragment/merged.fragments.aggregated.tsv.gz",
            skip_rows=0,
            has_header=False,
            separator="\t",
            use_pyarrow=False,
            new_columns=["Chromosome", "Start", "End", "Barcode", "Count", "Edit"])

df_fragments = df_fragments.drop("column_7")

df_fragments = df_fragments.filter(pl.col("Chromosome").is_in(["chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10",
                                                              "chr11","chr12","chr13","chr14","chr15","chr16","chr17","chr18","chr19",
                                                              "chrX","chrY"]))

In [10]:
df_fragments.head()

Chromosome,Start,End,Barcode,Count,Edit
str,i64,i64,str,i64,str
"""chr1""",3050003,3050371,"""GGATAGGAGTGCTAGG""",16,"""3050117|3050122|3050123|305012…"
"""chr1""",3050004,3050131,"""TTCATCAAGAACCCGA""",14,"""3050099|3050114|3050130"""
"""chr1""",3050004,3050149,"""TTACGTAAGCACCATA""",1,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGAACCATA""",12,"""3050116|3050117|3050130|305013…"
"""chr1""",3050004,3050149,"""TTACGTTAGCACCATT""",1,"""3050071|3050116|3050117|305013…"


In [11]:
df_meta = pd.read_csv("../../results/04_single_cell_access_atac/19_find_markers/marker_scores_per_cell.csv", 
                      index_col=0)
df_meta["barcode"] = df_meta.index
df_meta["barcode"] = df_meta["barcode"].str.split("#").str[1]

In [12]:
df_meta.head()

,Basal_Marker_Score_1,Club_Marker_Score_1,Ciliated_Marker_Score_1,Tuft_Marker_Score_1,Neuroendocrine_Marker_Score_1,Ionocyte_Marker_Score_1,Goblet_Marker_Score_1,barcode
mouse#GAGGCTCCAGAGTCGA,1.166078,0.166414,-0.004259,-0.017897,0.065168,0.116702,-0.116437,GAGGCTCCAGAGTCGA
mouse#CTGTATTTCGTATAGC,0.904784,0.745839,-0.006521,-0.045649,0.048694,0.034011,0.013150,CTGTATTTCGTATAGC
mouse#TATCGAGGTAACGTAA,1.075221,0.682136,-0.013413,-0.036434,0.036397,0.018903,-0.023247,TATCGAGGTAACGTAA
mouse#CGCAGGTAGAACGTCG,1.024818,0.795120,-0.015304,-0.023902,0.069732,0.035492,-0.040106,CGCAGGTAGAACGTCG
mouse#TTCTGTAGTATCCTTT,0.652378,1.233165,0.011927,-0.045276,0.068292,0.057492,0.071925,TTCTGTAGTATCCTTT


In [ ]:
cell_types = {
    "Ciliated": "Ciliated_Marker_Score_1",
    "Basal": "Basal_Marker_Score_1",
    "Club": "Club_Marker_Score_1",
    "Tuft": "Tuft_Marker_Score_1",
}

for cell_type, score_col in cell_types.items():
    df_selected = df_meta[df_meta[score_col] >= 1.0]
    selected_barcodes = set(df_selected["barcode"].tolist())

    print(f"{cell_type}: Selected {len(selected_barcodes)} barcodes with {score_col} >= 1.0")
    
    df_filtered_fragments = df_fragments.filter(pl.col("Barcode").is_in(selected_barcodes))
    
    output_path = os.path.join(out_dir, f"{cell_type}.cutoff_1.fragments.tsv")
    df_filtered_fragments.write_csv(output_path, separator="\t", include_header=False)
    sp.run(["bgzip", "-f", f"{out_dir}/{cell_type}.cutoff_1.fragments.tsv"])

Ciliated: Selected 12 barcodes with Ciliated_Marker_Score_1 >= 1.0
Basal: Selected 347 barcodes with Basal_Marker_Score_1 >= 1.0
Club: Selected 724 barcodes with Club_Marker_Score_1 >= 1.0
Tuft: Selected 38 barcodes with Tuft_Marker_Score_1 >= 1.0


In [ ]:
for cell_type, score_col in cell_types.items():
    df_selected = df_meta[df_meta[score_col] >= 0.95]
    selected_barcodes = set(df_selected["barcode"].tolist())

    print(f"{cell_type}: Selected {len(selected_barcodes)} barcodes with {score_col} >= 0.95")
    
    df_filtered_fragments = df_fragments.filter(pl.col("Barcode").is_in(selected_barcodes))
    
    output_path = os.path.join(out_dir, f"{cell_type}.cutoff_0.95.fragments.tsv")
    df_filtered_fragments.write_csv(output_path, separator="\t", include_header=False)
    sp.run(["bgzip", "-f", f"{out_dir}/{cell_type}.cutoff_0.95.fragments.tsv"])